# ۵. مدل‌سازی برد میزبان

رگرسیون لجستیک و جنگل تصادفی با تقسیم زمانی آموزش/آزمون مقایسه می‌شوند. سپس مزیت میزبانی در مسابقات غیرمساوی با Elo پیش از بازی تعدیل می‌شود.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "DataSet2").is_dir())
ANALYSIS_ROOT = ROOT / "UCL_Analysis2"
OUTPUT = ANALYSIS_ROOT / "Output"
sys.path.insert(0, str(ANALYSIS_ROOT / "src"))
pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-whitegrid")
print("Repository root:", ROOT)


In [ ]:
from modeling import run_and_save_models

metrics, importance, metadata, coefficients, adjusted = run_and_save_models(OUTPUT)
display(metrics.style.format({"accuracy": "{:.3f}", "balanced_accuracy": "{:.3f}", "roc_auc": "{:.3f}", "average_precision": "{:.3f}", "log_loss": "{:.3f}", "brier_score": "{:.3f}"}))
display(pd.Series(metadata, name="value").to_frame())


مدل‌ها بر اساس زمان جدا شده‌اند؛ ۸۰٪ مسابقات قدیمی برای آموزش و ۲۰٪ جدید برای آزمون‌اند. این کار از خوش‌بینی تقسیم تصادفی جلوگیری می‌کند. ویژگی‌های All-Time در این قسمت فقط یک تحلیل توصیفی/مقایسه‌ای هستند و نباید یک آزمون forecasting کاملاً leakage-free تلقی شوند.

In [ ]:
display(importance.style.format({"importance_mean_auc_decrease": "{:.4f}", "importance_sd": "{:.4f}"}))

fig, ax = plt.subplots(figsize=(9, 5))
plot_data = importance.sort_values("importance_mean_auc_decrease")
ax.barh(plot_data.feature, plot_data.importance_mean_auc_decrease, xerr=plot_data.importance_sd, color="#457b9d")
ax.axvline(0, color="black", lw=.8)
ax.set_xlabel("Decrease in held-out ROC-AUC after permutation")
ax.set_title("Random Forest permutation importance")
fig.tight_layout()
fig.savefig(OUTPUT / "model_feature_importance.png", dpi=180, bbox_inches="tight")
plt.show()


## برآورد تعدیل‌شده مزیت میزبانی

برای جدا کردن برد میزبان از مساوی، فقط مسابقات غیرمساوی استفاده می‌شوند. در این مدل، اختلاف Elo پیش از بازی، فصل و مرحله کنترل می‌شوند. عرض از مبدأ احتمال برد میزبان را برای دو تیم با Elo برابر نشان می‌دهد.

In [ ]:
display(pd.Series(adjusted, name="value").to_frame())
display(coefficients.style.format({
    "coefficient_log_odds": "{:.3f}", "robust_se": "{:.3f}", "p_value": "{:.3g}",
    "odds_ratio": "{:.3f}", "odds_ratio_ci_low": "{:.3f}", "odds_ratio_ci_high": "{:.3f}",
}))


## تفسیر درست

احتمال تعدیل‌شده این مدل مربوط به **سهم برد میزبان در مسابقات غیرمساوی** است، نه نرخ برد در تمام مسابقات. نرخ برد خام در تمام مسابقات در نوت‌بوک سوم گزارش شده است. همچنین این نتایج رابطه آماری‌اند و به‌تنهایی علیت را اثبات نمی‌کنند.